# Lab 18-08 · Power Cuts During Firmware Updates

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import bootsim as bs

## Parts 1 and 2 · Count the operations, then cut everywhere

*Predict* the number of flash operations and the number of bricking cuts for each scheme with six-sector images.

In [2]:
for s in ("inplace", "overwrite", "swap"):
    r = bs.campaign(s, 6)
    print(f"{s:9s}: {r['cuts']:3d} operations; new {r['new']:3d}, old {r['old']:3d}, bricked {r['bricked']:3d}")

inplace  :  14 operations; new   1, old   1, bricked  12
overwrite:  32 operations; new  19, old  13, bricked   0
swap     :  69 operations; new  55, old  14, bricked   0


## Part 3 · Break the bootloader

*Predict* which cuts brick a bootloader that resumes a swap only while the secondary slot still validates.

In [3]:
class BrokenResume(bs.Device):
    def boot(self):
        st = self.f.status
        if st is not None and st[0] == "pending":
            if bs.validate(self.slot(1)) is not None:
                if self.scheme == "swap":
                    self._swap_steps(st[1], st[2], st[3])
            return bs.validate(self.slot(0))
        return super().boot()

r = bs.campaign("swap", 6, device_cls=BrokenResume)
print("bricked", r["bricked"], "of", r["cuts"], "; first bricking operation (1-based):", r["bricking_cuts"][0] + 1)

bricked 44 of 69 ; first bricking operation (1-based): 20


## Part 4 · Revert

*Predict* what the next boot does when the new image never confirms itself, and whether cutting the revert can brick the device.

In [4]:
print("after an unconfirmed boot, the next boot runs version", bs.revert_test())
print(bs.revert_campaign())

after an unconfirmed boot, the next boot runs version 1
{'new': 0, 'old': 57, 'bricked': 0, 'cuts': 56}


## Part 5 · Budgets

*Predict* how installation time scales with image size and copies per sector.

In [5]:
sizes = np.array([128, 256, 384, 512, 768, 1024]) * 1024
for k, label in ((1, "overwrite"), (3, "swap via scratch")):
    plt.plot(sizes / 1024, [bs.swap_time(s, 4096, 20e-3, 20.48e-3, k) for s in sizes], "o-", label=label)
plt.xlabel("image size (KiB)"); plt.ylabel("installation time (s)"); plt.legend(); plt.show()
print("Example 18-08.4 risk per device:", bs.outage_probability(384 * 1024 / 50e3, 2))

Example 18-08.4 risk per device: 6.068129737024641e-06


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).